In [2]:
pip install -r requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import math
import os
import pandas as pd
import torch
import time as time
from omegaconf import OmegaConf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler

In [4]:
pd.set_option('display.max_columns', None)

In [ ]:
# Read pickles from the path specified in config.dest_file
config_path = "config.yaml"  # Replace with the path to your config file
config = OmegaConf.load(config_path)
df_encoded_path = os.path.join(config.dest_file, "df_encoded.csv")
# Load the DataFrame from the CSV file
df = pd.read_csv("df_encoded.csv")

In [6]:
# keep only 20% of the data for faster computation
df_small = df.sample(frac=1.0, random_state=42).reset_index(drop=False)

In [7]:
df_small.describe()

,index,surfaceProcessingLocationCavity,hotRunner,hotRunnerCost,manufacturingCost,mouldValidationCost,designTime,hourlyDesignCost,removedChassis,weightChassisProcessed,surfaceProcessingLocationChassis,removedCavity,numberOfCavities,weightCavityProcessed,weightMould,totalTransportationPercentage,percentageAircraft,percentageLorry,percentageTrain,percentageShip,totalDistance,transportCost,injectedMaterial_product,injectedMaterialCost,percentageRecycledMaterial,maxDepth,maxWallThickness,productVolume,materozzaVolume,nAnniProduzione,nProdottiAnno,materialDensity,tolerance,surfaceFinishing,cycleTime,machineCycleTime,maintenanceCost,productionCost,injectedMaterial_materozza,injectionMouldingProcess,memtiEngineValue,steelPrice,runnersType,mouldMaterialName,machineName,EUUSMacchina,CNMacchina,gateDiameter,setupTime,warmupTime,deliveryVolume,deliveryPeriod,mouldDesignCostDisplay,mouldTotalCost,Cost,human health - photochemical oxidation,ecosystem quality - terrestrial ecotoxicity,resources - mineral extraction,resources - non-renewable energy,ecosystem quality - terrestrial acidification & nutrification,resources - total,human health - ionising radiation,human health - respiratory effects (inorganics),human health - total,human health - human toxicity,ecosystem quality - aquatic ecotoxicity,climate change - climate change,human health - ozone layer depletion,ecosystem quality - land occupation,climate change - total,ecosystem quality - total
count,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.0000,164148.000000,164148.000000,164148.000000,164148.000000,1.641480e+05,164148.0,164148.0,164148.0,164148.0,164148.0,164148.000000,164148.0,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.0,164148.000000,164148.000000,164148.000000,164148.000000,164148.0,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.0,164148.000000,164148.0,164148.000000,164148.000000,164148.00000,164148.000000,164148.0,164148.000000,164148.0,164148.000000,164148.000000,164148.0,164148.0,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,164148.000000,1.641480e+05,164148.000000,164148.000000,164148.000000
mean,82073.500000,3.000000,0.195409,15862.270634,36722.073982,1606.586739,51.724541,50.532934,0.163086,6.6725,0.769793,0.308290,2.740917,0.005495,6.677995e+00,100.0,5.0,25.0,40.0,30.0,1133.808514,4000.0,3.455528,2.275922,0.461194,19.336300,1.319736,9.781782,0.319078,1.0,5381.814874,0.001102,0.147477,0.174690,360.0,14.347540,10185.344933,21.585642,3.501060,2.076979,0.0,0.426347,1.0,5.818901,9.061274,5.43275,4.115469,1.0,41.896764,30.0,6594.545654,3.644126,2000.0,38500.0,66584.271435,0.000040,0.003328,0.000024,0.030269,0.000185,0.030294,0.006191,0.014077,0.023236,0.002927,0.000106,0.019277,1.373095e-06,0.000173,0.019277,0.003792
std,47385.590331,0.554905,0.396516,10592.036270,12479.432960,505.081843,21.184073,2.776695,0.024592,0.0000,1.186966,0.037572,6.279052,0.000000,3.552725e-15,0.0,0.0,0.0,0.0,0.0,2229.091505,0.0,1.700838,2.623105,0.498493,29.477817,0.732969,9.831097,0.466121,0.0,15494.806796,0.000140,0.681254,0.768119,0.0,11.216855,6239.503454,11.208630,1.647026,0.525674,0.0,1.047509,0.0,1.404725,4.282591,3.29176,0.933774,0.0,38.959314,0.0,16912.852510,4.686030,0.0,0.0,15482.078690,0.000101,0.005769,0.000017,0.080793,0.000451,0.080809,0.015016,0.031402,0.051850,0.005354,0.000233,0.048485,3.047302e-06,0.000378,0.048485,0.006825
min,0.000000,2.000000,0.000000,10000.000000,25000.000000,1500.000000,40.000000,50.000000,0.150000,6.6725,0.000000,0.300000,1.000000,0.005495,6.677995e+00,100.0,5.0,25.0,40.0,30.0,50.000000,4000.0,0.000000,1.000000,0.000000,5.000000,1.000000,5.000000,0.000000,1.0,1.000000,0.000920,0.000000,0.000000,360.0,10.000000,8000.000000,12.000000,0.000000,1.000000,0.0,0.000000,1

From the above table, we know hat most of the target variables having distance correlation=1 are redundant or used for deriving the final variables. Hence it is good to eliminate them and keep only four ['human health - total', 'ecosystem quality - total', 'resources - total', 'Cost']

In [10]:
# Minimal VPBO for threshold and hidden size over four targets

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from typing import Callable, Optional, Tuple, List, Dict

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from scipy.optimize import minimize, Bounds
from joblib import Parallel, delayed
import dcor
import torch
import time

# ------------------------------
# data
# ------------------------------
# expects df_small already in memory, else load csv
try:
    _df = df_small.copy()
except NameError:
    _df = pd.read_csv("df_encoded.csv")

selected_features = [
    'surfaceProcessingLocationCavity','hotRunner','hotRunnerCost',
    'manufacturingCost','mouldValidationCost','designTime',
    'hourlyDesignCost','removedChassis','weightChassisProcessed',
    'surfaceProcessingLocationChassis','removedCavity','numberOfCavities',
    'weightCavityProcessed','weightMould','totalTransportationPercentage',
    'percentageAircraft','percentageLorry','percentageTrain','percentageShip',
    'totalDistance','injectedMaterial_product','injectedMaterialCost',
    'percentageRecycledMaterial','maxDepth','maxWallThickness',
    'productVolume','materozzaVolume','nAnniProduzione','nProdottiAnno',
    'materialDensity','tolerance','surfaceFinishing','cycleTime',
    'machineCycleTime','maintenanceCost','productionCost',
    'injectedMaterial_materozza','injectionMouldingProcess','memtiEngineValue',
    'steelPrice','runnersType','mouldMaterialName','machineName',
    'EUUSMacchina','CNMacchina','gateDiameter','setupTime','warmupTime',
    'deliveryVolume','deliveryPeriod','mouldDesignCostDisplay','mouldTotalCost'
]

output_variables = [
    'human health - total',
    'ecosystem quality - total',
    'resources - total',
    'Cost'
]

X_raw = _df[selected_features].values
Y_raw = _df[output_variables].values

sx = MinMaxScaler()
sy = MinMaxScaler()
X = sx.fit_transform(X_raw)
Y = sy.fit_transform(Y_raw)

n_targets = Y.shape[1]

# ------------------------------
# utilities
# ------------------------------

# ------------------------------
# evaluator
# ------------------------------
from src import mlp_eval_mean, BO

In [ ]:
# main.py
import os
import logging
from pathlib import Path
import numpy as np
from scipy.optimize import Bounds
import src as vpbo
from src import distmod_4



dim = 2
splits = 1
f_cores= 1

# make log folder
LOG_DIR = Path(f"log_folder_splits_{splits}_f_cores_{f_cores}")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / "run.log"

# root logging: console and file
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(), logging.FileHandler(LOG_FILE, encoding="utf-8")],
)

# route module logs to root
mod_logger = logging.getLogger(vpbo.__name__)  # 'src'
mod_logger.handlers.clear()
mod_logger.propagate = True
mod_logger.setLevel(logging.DEBUG)  # optional

# make log folder
LOG_DIR = Path(f"log_folder_splits_{splits}_f_cores_{f_cores}")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / "run.log"

# root logging: console and file
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(), logging.FileHandler(LOG_FILE, encoding="utf-8")],
)

# route module logs to root
mod_logger = logging.getLogger(vpbo.__name__)  # 'src'
mod_logger.handlers.clear()
mod_logger.propagate = True
mod_logger.setLevel(logging.DEBUG)  # optional

# ------------------------------
# run
# ------------------------------

best_hidden= []
best_thr= []
best_per_target_errors= []
best_dataset_shapes= []
start_time = time.time()
for i in range(len(output_variables)):
        dim = 2
        bounds = Bounds(lb=np.zeros(dim), ub=np.ones(dim))
        kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=np.ones(dim))
        bo = BO(distmod=mlp_eval_mean,
                args=(X, Y[:,i:i+1]), # only one target at a time
                dist_ref={'distrefmod': None},
                ref_args=(),
                dim=dim,
                bounds=bounds,
                kernel=kernel,
                exp_w=0.5,
                ub=np.ones(dim),
                lb=np.zeros(dim))

        lim_init = np.array([0.5, 0.5])
        bo.optimizer_vpbo(trials=10, split_num=1, lim_init=lim_init, f_cores=1, af_cores=1, ref_cores=1)
        np.random.seed(0)

        best_idx = np.argmin(np.mean(bo.y_vp, axis=1))
        
        best_thr = float(bo.x_vp[best_idx, 0])
        best_hidden = int(bo.x_vp[best_idx, 1] * 190 + 10)

        print("Best threshold:", round(best_thr, 3))

        print("Best hidden size:", best_hidden)

        print("Best per target errors:", bo.y_vp[best_idx])

        print("Dataset shapes:", bo.x_vp.shape, bo.y_vp.shape)


        # Save results to a text file
        with open(f'outputs/bo_results_target_{i}.txt', 'w') as f:
            f.write(f"Best threshold: {round(best_thr, 3)}\n")
            f.write(f"Best hidden size: {best_hidden}\n")
            f.write(f"Best per target errors: {bo.y_vp[best_idx]}\n")
            f.write(f"Dataset shapes: {bo.x_vp.shape}, {bo.y_vp.shape}\n")

end_time = time.time()
elapsed_time = end_time - start_time
print(f"Total elapsed time: {elapsed_time:.2f} seconds")

2025-09-12 17:38:32 INFO src optimizer_vpbo start trials=10 splits=1 dim=2
2025-09-12 17:38:32 DEBUG src random seed for splits created shape=(1, 2)
2025-09-12 17:38:32 DEBUG src mlp_eval_mean start batch=1
2025-09-12 17:38:32 INFO src mlp_eval start n=1 targets=1
2025-09-12 17:39:01 INFO src [Eval] thr=0.549, hidden=145, k=5, MRE=0.660457
2025-09-12 17:39:01 INFO src 
2025-09-12 17:39:01 INFO src mlp_eval done in 28.710s
2025-09-12 17:39:01 DEBUG src mlp_eval_mean done
2025-09-12 17:39:01 DEBUG src mlp_eval_mean start batch=1
2025-09-12 17:39:01 INFO src mlp_eval start n=1 targets=1
2025-09-12 17:39:29 INFO src [Eval] thr=0.500, hidden=105, k=5, MRE=0.496314
2025-09-12 17:39:29 INFO src 
2025-09-12 17:39:29 INFO src mlp_eval done in 27.609s
2025-09-12 17:39:29 DEBUG src mlp_eval_mean done
2025-09-12 17:39:29 INFO src initial eval done n=2
2025-09-12 17:39:29 DEBUG src ref_model_zero called batch=1 splits=1
2025-09-12 17:39:29 DEBUG src ref_model_zero called batch=1 splits=1
2025-09-12

Best threshold: 0.603
Best hidden size: 113
Best per target errors: [0.39662173]
Dataset shapes: (22, 2) (22, 1)


2025-09-12 17:46:53 INFO src [Eval] thr=0.549, hidden=145, k=5, MRE=1.175692
2025-09-12 17:46:53 INFO src 
2025-09-12 17:46:53 INFO src mlp_eval done in 28.118s
2025-09-12 17:46:53 DEBUG src mlp_eval_mean done
2025-09-12 17:46:53 DEBUG src mlp_eval_mean start batch=1
2025-09-12 17:46:53 INFO src mlp_eval start n=1 targets=1
2025-09-12 17:47:08 INFO src [Eval] thr=0.500, hidden=105, k=5, MRE=0.821062
2025-09-12 17:47:08 INFO src 
2025-09-12 17:47:08 INFO src mlp_eval done in 14.625s
2025-09-12 17:47:08 DEBUG src mlp_eval_mean done
2025-09-12 17:47:08 INFO src initial eval done n=2
2025-09-12 17:47:08 DEBUG src ref_model_zero called batch=1 splits=1
2025-09-12 17:47:08 DEBUG src ref_model_zero called batch=1 splits=1
2025-09-12 17:47:08 INFO src GP models initialized count=1
2025-09-12 17:47:08 DEBUG src restarts per split=32
2025-09-12 17:47:08 DEBUG src LCB_AF value=0.909720 mu=0.998377 std=0.177314 yref=0.000000
2025-09-12 17:47:08 DEBUG src LCB_AF value=0.909720 mu=0.998377 std=0.177

Best threshold: 0.055
Best hidden size: 67
Best per target errors: [0.06498296]
Dataset shapes: (22, 2) (22, 1)


2025-09-12 17:52:23 INFO src [Eval] thr=0.549, hidden=145, k=5, MRE=0.519068
2025-09-12 17:52:23 INFO src 
2025-09-12 17:52:23 INFO src mlp_eval done in 15.418s
2025-09-12 17:52:23 DEBUG src mlp_eval_mean done
2025-09-12 17:52:23 DEBUG src mlp_eval_mean start batch=1
2025-09-12 17:52:23 INFO src mlp_eval start n=1 targets=1
2025-09-12 17:52:38 INFO src [Eval] thr=0.500, hidden=105, k=5, MRE=1.136222
2025-09-12 17:52:38 INFO src 
2025-09-12 17:52:38 INFO src mlp_eval done in 14.662s
2025-09-12 17:52:38 DEBUG src mlp_eval_mean done
2025-09-12 17:52:38 INFO src initial eval done n=2
2025-09-12 17:52:38 DEBUG src ref_model_zero called batch=1 splits=1
2025-09-12 17:52:38 DEBUG src ref_model_zero called batch=1 splits=1
2025-09-12 17:52:38 INFO src GP models initialized count=1
2025-09-12 17:52:38 DEBUG src restarts per split=32
2025-09-12 17:52:38 DEBUG src LCB_AF value=0.673357 mu=0.827645 std=0.308577 yref=0.000000
2025-09-12 17:52:38 DEBUG src LCB_AF value=0.673357 mu=0.827645 std=0.308

Best threshold: 0.055
Best hidden size: 67
Best per target errors: [0.28499107]
Dataset shapes: (22, 2) (22, 1)


2025-09-12 17:57:51 INFO src [Eval] thr=0.549, hidden=145, k=5, MRE=0.027967
2025-09-12 17:57:51 INFO src 
2025-09-12 17:57:51 INFO src mlp_eval done in 15.411s
2025-09-12 17:57:51 DEBUG src mlp_eval_mean done
2025-09-12 17:57:51 DEBUG src mlp_eval_mean start batch=1
2025-09-12 17:57:51 INFO src mlp_eval start n=1 targets=1
2025-09-12 17:58:05 INFO src [Eval] thr=0.500, hidden=105, k=5, MRE=0.020341
2025-09-12 17:58:05 INFO src 
2025-09-12 17:58:05 INFO src mlp_eval done in 14.512s
2025-09-12 17:58:05 DEBUG src mlp_eval_mean done
2025-09-12 17:58:05 INFO src initial eval done n=2
2025-09-12 17:58:05 DEBUG src ref_model_zero called batch=1 splits=1
2025-09-12 17:58:05 DEBUG src ref_model_zero called batch=1 splits=1
2025-09-12 17:58:05 INFO src GP models initialized count=1
2025-09-12 17:58:05 DEBUG src restarts per split=32
2025-09-12 17:58:05 DEBUG src LCB_AF value=0.022248 mu=0.024154 std=0.003813 yref=0.000000
2025-09-12 17:58:05 DEBUG src LCB_AF value=0.022248 mu=0.024154 std=0.003

Best threshold: 0.055
Best hidden size: 67
Best per target errors: [0.00128627]
Dataset shapes: (22, 2) (22, 1)
Total elapsed time: 1467.19 seconds


Exception ignored in: <function ResourceTracker.__del__ at 0x7f687bfdaac0>
Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/conda/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/conda/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7f1489e66ac0>
Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/conda/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/conda/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7f5c283faac0>
Traceback (most recent call last):
  File "/opt